# Feature Engineering

Builds modeling features on top of the raw `Match` table in `data/database.sqlite`.

Feature engineering cited from https://pubmed.ncbi.nlm.nih.gov/40989473/

Run `python scripts/download_data.py` from the repo root first so that `data/database.sqlite` exists.

First feature: the match result label from the home team's perspective — `Win` / `Draw` / `Defeat`. This is the target variable for the outcome predictor.

Second feature: overall player rating. For each player, use the closest rating from the FIFA metrics as the representative value.

Third feature: team strength. Evaluate both teams on overall strength based on an aggregation of the player ratings.

In [64]:
import sqlite3
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DB_PATH = REPO_ROOT / "data" / "database.sqlite"
assert DB_PATH.exists(), (
    f"Missing {DB_PATH}. Run `python scripts/download_data.py` from the repo root."
)

conn = sqlite3.connect(DB_PATH)
print(f"Opened {DB_PATH} ({DB_PATH.stat().st_size / 1024**2:.1f} MB)")

Opened /Users/alexy/CSE/Sports-NLP-Outcome-Predictor/data/database.sqlite (298.6 MB)


## Load matches

Load the columns needed for the result label plus `match_api_id` and the 22 player ID
columns required for the player-rating lookup in the next section.
The full table is 115 columns wide; we keep only what we need.

In [65]:
matches = pd.read_sql(
    """
    SELECT match_api_id, date,
           home_team_api_id, away_team_api_id,
           home_team_goal, away_team_goal,
           home_player_1,  home_player_2,  home_player_3,
           home_player_4,  home_player_5,  home_player_6,
           home_player_7,  home_player_8,  home_player_9,
           home_player_10, home_player_11,
           away_player_1,  away_player_2,  away_player_3,
           away_player_4,  away_player_5,  away_player_6,
           away_player_7,  away_player_8,  away_player_9,
           away_player_10, away_player_11
    FROM Match
    ORDER BY date;
    """,
    conn,
)
print(f"Loaded {len(matches):,} matches")
matches.head()

Loaded 25,979 matches


,match_api_id,date,home_team_api_id,away_team_api_id,home_team_goal,away_team_goal,home_player_1,home_player_2,home_player_3,home_player_4,home_player_5,home_player_6,home_player_7,home_player_8,home_player_9,home_player_10,home_player_11,away_player_1,away_player_2,away_player_3,away_player_4,away_player_5,away_player_6,away_player_7,away_player_8,away_player_9,away_player_10,away_player_11
0,486263,2008-07-18 00:00:00,10192,9931,1,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,486264,2008-07-19 00:00:00,9930,10179,3,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,486265,2008-07-20 00:00:00,10199,9824,1,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,486266,2008-07-20 00:00:00,7955,10243,1,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,486267,2008-07-23 00:00:00,9931,9956,1,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Add the `result` column

Labels are from the home team's perspective:

- Win: home goals > away goals
- Draw: home goals == away goals
- Defeat: home goals < away goals

This matches how the rest of the dataset is structured (e.g. the bookmaker odds columns `B365H/D/A` are also home-perspective), so the label lines up cleanly with the other features.

In [66]:
conditions = [
    matches["home_team_goal"] > matches["away_team_goal"],
    matches["home_team_goal"] == matches["away_team_goal"],
    matches["home_team_goal"] < matches["away_team_goal"],
]
matches["result"] = np.select(conditions, ["Win", "Draw", "Defeat"], default="")

# Integer encoding for sklearn-style models
matches["result_code"] = matches["result"].map({"Defeat": 0, "Draw": 1, "Win": 2})

matches[["date", "home_team_goal", "away_team_goal", "result", "result_code"]].head(10)

,date,home_team_goal,away_team_goal,result,result_code
0,2008-07-18 00:00:00,1,2,Defeat,0
1,2008-07-19 00:00:00,3,1,Win,2
2,2008-07-20 00:00:00,1,2,Defeat,0
3,2008-07-20 00:00:00,1,2,Defeat,0
4,2008-07-23 00:00:00,1,0,Win,2
5,2008-07-23 00:00:00,1,2,Defeat,0
6,2008-07-23 00:00:00,1,0,Win,2
7,2008-07-24 00:00:00,2,1,Win,2
8,2008-07-24 00:00:00,0,2,Defeat,0
9,2008-07-26 00:00:00,2,0,Win,2


## Sanity checks

Confirm no rows fell through the conditions and that the class distribution shows the expected home-field advantage (~46% Win / 25% Draw / 29% Defeat for European league football).

In [67]:
assert matches["result"].notna().all(), "Some matches were not labeled"

print("Class counts:")
print(matches["result"].value_counts())
print("\nClass proportions:")
print(matches["result"].value_counts(normalize=True).round(3))

Class counts:
result
Win       11917
Defeat     7466
Draw       6596
Name: count, dtype: int64

Class proportions:
result
Win       0.459
Defeat    0.287
Draw      0.254
Name: proportion, dtype: float64


## Obtain player ratings

Melt the 22 player ID columns into long format, then merge against Player_Attributes to obtain the most recent rating before the match date.

In [68]:
# define player id columns
player_cols = [f'home_player_{i}' for i in range(1, 12)] + \
              [f'away_player_{i}' for i in range(1, 12)]

# melt 22 player id columns into one row per player per match
match_players = (
    matches[['match_api_id', 'date'] + player_cols]
    .melt(id_vars=['match_api_id', 'date'], value_vars=player_cols, value_name='player_api_id')
    .dropna(subset=['player_api_id'])
    .astype({'player_api_id': int})
)

# load player attributes
player_attributes = pd.read_sql("SELECT player_api_id, date, overall_rating FROM Player_Attributes", conn)

# sort by date, required for merge_asof
match_players['date'] = pd.to_datetime(match_players['date'])
player_attributes['date'] = pd.to_datetime(player_attributes['date'])

match_players = match_players.sort_values('date')
player_attributes = player_attributes.sort_values('date')

# get the most recent rating before each match date
rated = pd.merge_asof(
    match_players,
    player_attributes,
    on='date',
    by='player_api_id',
    direction='backward'
)

## Build match_team_stats

Step 2: aggregate rated by match and side into team-level stats.

Step 3: merge back onto matches so each match row has home and away rating features.

In [69]:
# extract side (home or away) from the variable column
rated['side'] = rated['variable'].str.split('_player_').str[0]

match_team_stats = (
    rated.groupby(['match_api_id', 'side'])['overall_rating']
    .agg(
        avg_rating='mean',
        min_rating='min',
        max_rating='max',
        std_rating='std',
    )
    .unstack('side')
)

# flatten MultiIndex columns, e.g. (avg_rating, home) becomes home_avg_rating
match_team_stats.columns = [
    f'{side}_{stat}' for stat, side in match_team_stats.columns
]
match_team_stats = match_team_stats.reset_index()

print(match_team_stats.shape)
match_team_stats.head()

(25221, 9)


,match_api_id,away_avg_rating,home_avg_rating,away_min_rating,home_min_rating,away_max_rating,home_max_rating,away_std_rating,home_std_rating
0,483129,69.200000,70.000000,61.0,51.0,81.0,81.0,6.250333,8.921883
1,483130,69.454545,74.454545,58.0,65.0,77.0,83.0,5.354692,5.905313
2,483131,68.818182,64.909091,55.0,47.0,75.0,77.0,7.110811,9.300049
3,483132,67.181818,70.444444,58.0,61.0,75.0,76.0,4.707827,5.270463
4,483133,70.363636,78.818182,58.0,74.0,79.0,83.0,7.131237,3.250175


In [70]:
# merge team stats back onto the matches dataframe
matches = matches.merge(match_team_stats, on='match_api_id', how='left')

print(matches.shape)
matches[['match_api_id', 'result', 'home_avg_rating', 'away_avg_rating']].head(10)

(25979, 38)


,match_api_id,result,home_avg_rating,away_avg_rating
0,486263,Defeat,NaN,NaN
1,486264,Win,NaN,NaN
2,486265,Defeat,NaN,NaN
3,486266,Defeat,NaN,NaN
4,486267,Win,NaN,NaN
5,486268,Defeat,NaN,NaN
6,486269,Win,NaN,NaN
7,486270,Win,NaN,NaN
8,486271,Defeat,NaN,NaN
9,486272,Win,NaN,NaN


Some matches have no player lineup data recorded in the database, so they produce NaN ratings after the merge. These are dropped before modelling.

In [71]:
null_rate = matches['home_avg_rating'].isna().mean()
print(f"Matches with no rating data: {null_rate:.1%}")
print(f"Total matches: {len(matches)}, missing: {matches['home_avg_rating'].isna().sum()}")

matches = matches.dropna(subset=['home_avg_rating', 'away_avg_rating'])
print(f'{len(matches):,} matches retained with full rating data')

Matches with no rating data: 2.9%
Total matches: 25979, missing: 758
25,221 matches retained with full rating data


## Team performance statistics (rolling form)

For each match, compute each team's average goals scored, goals conceded, and points per game over their last 5 matches. Shift by 1 so the current match does not leak into the feature.

In [72]:
FORM_WINDOW = 5

def compute_team_form(df, n=FORM_WINDOW):
    """Rolling per-team form stats, leak-free (shift 1 before rolling mean)."""
    home = df[['match_api_id', 'date', 'home_team_api_id', 'home_team_goal', 'away_team_goal']].copy()
    home.columns = ['match_api_id', 'date', 'team_id', 'goals_for', 'goals_against']
    home['is_home'] = True

    away = df[['match_api_id', 'date', 'away_team_api_id', 'away_team_goal', 'home_team_goal']].copy()
    away.columns = ['match_api_id', 'date', 'team_id', 'goals_for', 'goals_against']
    away['is_home'] = False

    timeline = pd.concat([home, away], ignore_index=True).sort_values(['team_id', 'date'])
    timeline['points'] = np.select(
        [timeline['goals_for'] > timeline['goals_against'],
         timeline['goals_for'] == timeline['goals_against']],
        [3, 1], default=0
    )

    for stat in ['goals_for', 'goals_against', 'points']:
        timeline[f'rolling_{stat}'] = (
            timeline.groupby('team_id')[stat]
            .transform(lambda x: x.shift(1).rolling(n, min_periods=1).mean())
        )

    rename_home = {'rolling_goals_for': 'home_rolling_gf',
                   'rolling_goals_against': 'home_rolling_ga',
                   'rolling_points': 'home_rolling_pts'}
    rename_away = {'rolling_goals_for': 'away_rolling_gf',
                   'rolling_goals_against': 'away_rolling_ga',
                   'rolling_points': 'away_rolling_pts'}

    home_form = (timeline[timeline['is_home']]
                 .rename(columns=rename_home)
                 [['match_api_id', 'home_rolling_gf', 'home_rolling_ga', 'home_rolling_pts']])
    away_form = (timeline[~timeline['is_home']]
                 .rename(columns=rename_away)
                 [['match_api_id', 'away_rolling_gf', 'away_rolling_ga', 'away_rolling_pts']])

    return home_form.merge(away_form, on='match_api_id')


team_form = compute_team_form(matches)
print(team_form.shape)
team_form.head()

(25221, 7)


,match_api_id,home_rolling_gf,home_rolling_ga,home_rolling_pts,away_rolling_gf,away_rolling_ga,away_rolling_pts
0,506666,0.00,0.00,1.0,1.0,0.000000,2.000000
1,506688,0.75,0.75,1.0,0.0,1.666667,0.333333
2,506704,0.60,0.80,0.8,1.0,1.250000,1.500000
3,506717,0.80,1.00,0.8,1.8,0.400000,2.600000
4,506736,0.40,1.20,0.4,0.8,0.800000,1.400000


## Head-to-head history

For each match, look back at the last 5 meetings between the same pair of teams (regardless of home/away) and compute the home team's win rate and average goal differential in those prior meetings.

The loop is O(n²) in the worst case but manageable for ~25k matches; rows with no prior H2H history are kept as NaN and can be filled with league averages or dropped at modelling time.

In [73]:
H2H_WINDOW = 5

def compute_h2h(df, n=H2H_WINDOW):
    """Per-match H2H win rate and avg goal diff from the home team's perspective."""
    df_sorted = df.sort_values('date').reset_index(drop=True)

    # Canonical pair key so (A,B) and (B,A) are treated as the same fixture
    df_sorted['_t1'] = np.minimum(df_sorted['home_team_api_id'], df_sorted['away_team_api_id'])
    df_sorted['_t2'] = np.maximum(df_sorted['home_team_api_id'], df_sorted['away_team_api_id'])

    records = []
    for idx, row in df_sorted.iterrows():
        prior = df_sorted.iloc[:idx]
        prior = prior[(prior['_t1'] == row['_t1']) & (prior['_t2'] == row['_t2'])].tail(n)

        if prior.empty:
            records.append({'match_api_id': row['match_api_id'],
                            'h2h_home_win_rate': np.nan,
                            'h2h_avg_goal_diff': np.nan,
                            'h2h_n_matches': 0})
            continue

        h = row['home_team_api_id']
        home_perspective_diff = np.where(
            prior['home_team_api_id'] == h,
            prior['home_team_goal'] - prior['away_team_goal'],
            prior['away_team_goal'] - prior['home_team_goal'],
        )
        wins = (home_perspective_diff > 0).sum()

        records.append({'match_api_id': row['match_api_id'],
                        'h2h_home_win_rate': wins / len(prior),
                        'h2h_avg_goal_diff': home_perspective_diff.mean(),
                        'h2h_n_matches': len(prior)})

    return pd.DataFrame(records).drop(columns=['_t1', '_t2'], errors='ignore')


print("Computing H2H features (may take ~30 s)…")
h2h = compute_h2h(matches)
print(h2h.shape)
h2h.head(10)

Computing H2H features (may take ~30 s)…
(25221, 4)


,match_api_id,h2h_home_win_rate,h2h_avg_goal_diff,h2h_n_matches
0,483129,NaN,NaN,0
1,483130,NaN,NaN,0
2,483131,NaN,NaN,0
3,483132,NaN,NaN,0
4,483134,NaN,NaN,0
5,483135,NaN,NaN,0
6,483136,NaN,NaN,0
7,483137,NaN,NaN,0
8,483138,NaN,NaN,0
9,489981,NaN,NaN,0


## Team Attributes (FIFA tactical ratings)

The `Team_Attributes` table contains per-team tactical snapshots at various dates. Use `merge_asof` (same approach as player ratings) to attach the most recent snapshot before each match for both the home and away team.

In [74]:
conn = sqlite3.connect(DB_PATH)

TACTICAL_COLS = [
    'buildUpPlaySpeed', 'buildUpPlayPassing',
    'chanceCreationPassing', 'chanceCreationCrossing', 'chanceCreationShooting',
    'defencePressure', 'defenceAggression', 'defenceTeamWidth',
]

team_attrs = pd.read_sql(
    f"SELECT team_api_id, date, {', '.join(TACTICAL_COLS)} FROM Team_Attributes",
    conn,
)
conn.close()

team_attrs['date'] = pd.to_datetime(team_attrs['date'])
team_attrs = team_attrs.sort_values('date')

matches['date'] = pd.to_datetime(matches['date'])
matches_sorted = matches.sort_values('date')

def merge_team_attrs(matches_df, team_id_col, prefix):
    merged = pd.merge_asof(
        matches_df[['match_api_id', 'date', team_id_col]],
        team_attrs,
        on='date',
        left_by=team_id_col,
        right_by='team_api_id',
        direction='backward',
    ).drop(columns=['team_api_id', team_id_col])
    return merged.rename(columns={c: f'{prefix}_{c}' for c in TACTICAL_COLS})

home_attrs = merge_team_attrs(matches_sorted, 'home_team_api_id', 'home')
away_attrs = merge_team_attrs(matches_sorted, 'away_team_api_id', 'away')

print("home_attrs:", home_attrs.shape)
print("away_attrs:", away_attrs.shape)
home_attrs.head()

home_attrs: (25221, 10)
away_attrs: (25221, 10)


,match_api_id,date,home_buildUpPlaySpeed,home_buildUpPlayPassing,home_chanceCreationPassing,home_chanceCreationCrossing,home_chanceCreationShooting,home_defencePressure,home_defenceAggression,home_defenceTeamWidth
0,483129,2008-08-09,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,483130,2008-08-09,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,483131,2008-08-09,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,483132,2008-08-09,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,483134,2008-08-09,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Final merge

Combine all three feature sets with the base match table on `match_api_id` to produce the complete modelling dataset.

In [75]:
RATING_COLS = [
    'home_avg_rating', 'away_avg_rating',
    'home_min_rating', 'away_min_rating',
    'home_max_rating', 'away_max_rating',
    'home_std_rating', 'away_std_rating',
]

home_tactical_cols = [f'home_{c}' for c in TACTICAL_COLS]
away_tactical_cols = [f'away_{c}' for c in TACTICAL_COLS]

dataset = (
    matches[['match_api_id', 'date', 'result', 'result_code'] + RATING_COLS]
    .merge(team_form,  on='match_api_id', how='left')
    .merge(h2h,        on='match_api_id', how='left')
    .merge(home_attrs[['match_api_id'] + home_tactical_cols], on='match_api_id', how='left')
    .merge(away_attrs[['match_api_id'] + away_tactical_cols], on='match_api_id', how='left')
)

print(f"Dataset shape: {dataset.shape}")
print(f"\nNull rates:\n{dataset.isnull().mean().round(3).to_string()}")
dataset.head()

Dataset shape: (25221, 37)

Null rates:
match_api_id                   0.000
date                           0.000
result                         0.000
result_code                    0.000
home_avg_rating                0.000
away_avg_rating                0.000
home_min_rating                0.000
away_min_rating                0.000
home_max_rating                0.000
away_max_rating                0.000
home_std_rating                0.000
away_std_rating                0.000
home_rolling_gf                0.006
home_rolling_ga                0.006
home_rolling_pts               0.006
away_rolling_gf                0.006
away_rolling_ga                0.006
away_rolling_pts               0.006
h2h_home_win_rate              0.138
h2h_avg_goal_diff              0.138
h2h_n_matches                  0.000
home_buildUpPlaySpeed          0.210
home_buildUpPlayPassing        0.210
home_chanceCreationPassing     0.210
home_chanceCreationCrossing    0.210
home_chanceCreationShooting    0.21

,match_api_id,date,result,result_code,home_avg_rating,away_avg_rating,home_min_rating,away_min_rating,home_max_rating,away_max_rating,home_std_rating,away_std_rating,home_rolling_gf,home_rolling_ga,home_rolling_pts,away_rolling_gf,away_rolling_ga,away_rolling_pts,h2h_home_win_rate,h2h_avg_goal_diff,h2h_n_matches,home_buildUpPlaySpeed,home_buildUpPlayPassing,home_chanceCreationPassing,home_chanceCreationCrossing,home_chanceCreationShooting,home_defencePressure,home_defenceAggression,home_defenceTeamWidth,away_buildUpPlaySpeed,away_buildUpPlayPassing,away_chanceCreationPassing,away_chanceCreationCrossing,away_chanceCreationShooting,away_defencePressure,away_defenceAggression,away_defenceTeamWidth
0,483129,2008-08-09,Win,2,70.000000,69.200000,51.0,61.0,81.0,81.0,8.921883,6.250333,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,483130,2008-08-09,Win,2,74.454545,69.454545,65.0,58.0,83.0,77.0,5.905313,5.354692,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,483131,2008-08-09,Win,2,64.909091,68.818182,47.0,55.0,77.0,75.0,9.300049,7.110811,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,483132,2008-08-09,Defeat,0,70.444444,67.181818,61.0,58.0,76.0,75.0,5.270463,4.707827,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,483134,2008-08-09,Win,2,67.909091,73.363636,52.0,51.0,76.0,85.0,6.789029,9.972690,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [76]:
conn.close()